# Handle large datasets with BigQuery — Self-learning

**Data Analytics · Optional hands-on**

**Learning goals:** explore public data; estimate query processing; use column/partition/clustering filters; understand JOIN cardinality; run GoogleSQL from Python.

This notebook is optional. **You can study every example without executing any code or creating a Google Cloud account.** The companion presentation provides the architectural background.


## 0. Before you start — optional activity, costs and security

**This activity is optional.** Creating a Google Cloud account, enabling billing and executing queries are **not required**. Ironhack does not provide, manage or fund Google Cloud accounts or projects for this optional activity. If you prefer not to use your own account, **read the notebook without running the code**.

If you voluntarily use your own account, you are responsible for managing its usage and billing settings and for any associated charges. **Ironhack does not reimburse charges incurred through personal Google Cloud accounts for this optional activity.** This notice is subject to Ironhack's applicable policies and legal review.

**Google Cloud prices, free allowances, quotas, budgets and billing conditions can change.** Review the official information before using the service:

- [Get started with Google Cloud](https://cloud.google.com/docs/get-started) · [BigQuery quickstarts](https://cloud.google.com/bigquery/docs/quickstarts) · [BigQuery Sandbox](https://cloud.google.com/bigquery/docs/sandbox) (features and limits apply).
- [BigQuery pricing](https://cloud.google.com/bigquery/pricing) · [Quotas](https://cloud.google.com/bigquery/quotas) · [Budgets and alerts](https://cloud.google.com/billing/docs/how-to/budgets).
- Protect your account with [two-step verification](https://support.google.com/accounts/answer/185839).

**Budgets and alerts are not hard spending caps.** Query dry runs estimate processing, not monetary cost; `maximum_bytes_billed` is a per-query protection for on-demand pricing, **not** an account-wide cost limit. Other services or storage may incur charges.

We query Google's public datasets without copying their tables, so their **original storage is not billed to your project**. Your project may still be charged for query processing. Avoid creating datasets or tables unless you understand the costs.


### 0.1. Conda environment (terminal, not notebook)

```bash
conda create -n bigquery-learning python=3.11 -y
conda activate bigquery-learning
conda install -c conda-forge google-cloud-bigquery pandas pandas-gbq pyarrow jupyterlab ipykernel -y
python -m ipykernel install --user --name bigquery-learning --display-name "Python (BigQuery)"
```

Select the **Python (BigQuery)** kernel. The Google Cloud CLI is installed **separately**, following the [official instructions](https://cloud.google.com/sdk/docs/install). Do not paste terminal authentication commands into notebook cells.

### 0.2. Authenticate securely with Application Default Credentials (ADC)

In your **terminal** (not in a notebook):

```bash
gcloud auth application-default login
gcloud auth application-default set-quota-project YOUR_PROJECT_ID
```

Follow the browser sign-in flow. The CLI saves local credentials in its standard configuration directory; the Python client discovers them automatically. A downloaded service-account JSON key is **not required** for this personal learning exercise.

**Finding `YOUR_PROJECT_ID`:** open the [Google Cloud Console](https://console.cloud.google.com/), select your project in the top project selector, and copy **Project ID** (not Project Name or Project Number). Replace the placeholder below.

**Never print, display, paste or commit credentials, access tokens, API keys, or credential JSON.** Notebook **outputs are saved inside `.ipynb` files**. Do not inspect the ADC credential file in the notebook. Clear outputs before sharing/committing; inspect code **and outputs** for secrets. If a credential was exposed, revoke/rotate it; clearing outputs does not erase Git history. Do not commit `.env` or credential files.

### 0.3. Python setup and query safety helpers

**Run only after configuring ADC and your project.** Replace `YOUR_PROJECT_ID` with the **Project ID** shown in the [Google Cloud Console](https://console.cloud.google.com/) project selector (not the display name or project number).

Our safety workflow is: **write SQL → dry run → inspect estimated bytes → execute only if acceptable, with `maximum_bytes_billed`.** We use a conservative 1 GiB example cap. Do not raise it just to make a query run. A dry run is not a price quotation, and clustered-table estimates may be upper bounds.

The `RUN_QUERIES` switch defaults to `False`. This lets you run all Python cells to inspect the SQL **without submitting jobs**. Change it to `True` only after reviewing your account, the estimate, and the per-query limit. Dry runs themselves require an authenticated project but do not execute the SQL.


In [ ]:
from google.cloud import bigquery

PROJECT_ID = "YOUR_PROJECT_ID"  # Replace with your own Google Cloud Project ID
MAX_BYTES_BILLED = 1 * 1024**3  # 1 GiB per on-demand query
RUN_DRY_RUNS = False  # Set True only after configuring ADC and reviewing billing
RUN_QUERIES = False    # Set True only when you explicitly choose to execute queries

# The client uses local Application Default Credentials (ADC); no secrets in this notebook.
client = None
if RUN_DRY_RUNS or RUN_QUERIES:
    if PROJECT_ID == "YOUR_PROJECT_ID":
        raise ValueError("Replace YOUR_PROJECT_ID before using BigQuery.")
    client = bigquery.Client(project=PROJECT_ID)


def estimate_query_cost(client, sql):
    """Perform a BigQuery dry run and return estimated bytes processed."""
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=config)
    estimated_bytes = job.total_bytes_processed
    print(f"Estimated data processed: {estimated_bytes / 1024**3:.3f} GiB")
    print("Estimate of bytes processed; not an exact monetary price.")
    return estimated_bytes


def run_query_safely(client, sql, max_bytes_billed=MAX_BYTES_BILLED):
    """Estimate first, then execute only when the estimate fits the cap."""
    estimated_bytes = estimate_query_cost(client, sql)
    if estimated_bytes > max_bytes_billed:
        raise ValueError("Estimated bytes exceed the cap. Query not executed.")
    config = bigquery.QueryJobConfig(maximum_bytes_billed=max_bytes_billed)
    return client.query(sql, job_config=config).result()


def review_query(label, sql, *, execute=False):
    """Show SQL; optionally dry-run and execute it under the configured cap."""
    print(f"\n--- {label} ---\n{sql.strip()}\n")
    if not RUN_DRY_RUNS:
        print("Dry run disabled. Review the SQL without using Google Cloud.")
        return None
    estimated_bytes = estimate_query_cost(client, sql)
    if execute and RUN_QUERIES:
        if estimated_bytes > MAX_BYTES_BILLED:
            print("Skipped: estimate exceeds MAX_BYTES_BILLED.")
            return None
        # Enforce the limit server-side too, even if the estimate is acceptable.
        config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED)
        return client.query(sql, job_config=config).result()
    print("Query not executed.")
    return None


## 1. From MySQL to BigQuery

MySQL is commonly used for **transactional** workloads (OLTP); BigQuery is a managed, distributed **analytical** data warehouse (OLAP). BigQuery stores data in a column-oriented format and separates storage and compute. Scaling up a single server is not the only option: analytical engines distribute work across resources.

**Trade-off:** BigQuery is optimized for large analytical scans, not low-latency row-by-row transactions. JOINs **are supported**; performance depends on query design and data cardinality. BigQuery supports denormalized and normalized schemas, including nested/repeated fields; denormalization is a choice, not a requirement.

See the companion presentation for architecture diagrams. [Official overview](https://cloud.google.com/bigquery/docs/introduction).

## 2. Explore the BigQuery console

Open [BigQuery in Google Cloud Console](https://console.cloud.google.com/bigquery). Select **your own project** in the project selector. In **Explorer**, locate or add the public project `bigquery-public-data` (use the **Add data** / **Star project** options if available; UI labels may change).

A fully qualified table name has the form `project.dataset.table`. Here is a small SQL query to try later:

```sql
SELECT 'hello' AS message;
```

This constant-expression query does not scan a table. Run it in the SQL editor only if your project is ready. Explore **Schema**, **Preview** and **Details** tabs of a public table. Preview is a useful way to inspect sample records without writing `SELECT *`.

**No personal dataset is required for the read-only exercises below.** We intentionally avoid creating copies of large public tables.

## 3. Column pruning and the `LIMIT` trap — Chicago Taxi Trips

We will compare three SQL queries on `bigquery-public-data.chicago_taxi_trips.taxi_trips`.

**A and B are for dry-run estimates only: do not execute them.** A `LIMIT` generally reduces returned rows, **not** bytes scanned on a non-clustered table; clustered tables can be an exception. Query C selects only two columns, reducing the amount of data scanned. The public dataset and estimates may change over time.

**Exercise:** inspect the three queries below and predict which scans fewer bytes. Enable `RUN_DRY_RUNS` only if you have configured your own account. Do not execute the full-table scans.

[Cost best practices](https://cloud.google.com/bigquery/docs/best-practices-costs).


In [ ]:
TAXI_FULL = """
SELECT *
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
"""
TAXI_LIMIT = """
SELECT *
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
LIMIT 10
"""
TAXI_COLUMNS = """
SELECT payment_type, fare
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
LIMIT 10
"""

# Show each SQL query, then dry-run it if enabled. Never execute A or B.
review_query("A — all columns (estimate only)", TAXI_FULL)
review_query("B — all columns with LIMIT (estimate only)", TAXI_LIMIT)
review_query("C — two columns (estimate only)", TAXI_COLUMNS)

# C may still exceed the conservative 1 GiB cap; do not raise the cap automatically.
# To execute a different, smaller query, see the constant-expression example below.


## 4. Partition pruning and clustering — Wikipedia Pageviews

BigQuery can **partition** tables by date and **cluster** data by selected columns. Partition filters can skip partitions; clustering filters can help skip storage blocks. Actual billed bytes can differ from a dry-run estimate for clustered tables.

The companion presentation uses `bigquery-public-data.wikipedia.pageviews_2025`. **Check the table's current availability, schema and partition/clustering settings** in the console; public datasets can change.

The next cell defines three SQL queries: the full table, one day, and one day with `wiki = 'en'`. **Compare dry-run estimates only; do not execute these scans as part of the exercise.**

[Partition pruning](https://cloud.google.com/bigquery/docs/querying-partitioned-tables) · [Clustered tables](https://cloud.google.com/bigquery/docs/clustered-tables).


In [ ]:
WIKI_ALL = """
SELECT wiki, datehour, views
FROM `bigquery-public-data.wikipedia.pageviews_2025`
"""
WIKI_DAY = """
SELECT wiki, datehour, views
FROM `bigquery-public-data.wikipedia.pageviews_2025`
WHERE DATE(datehour) = DATE '2025-03-30'
"""
WIKI_DAY_EN = """
SELECT wiki, datehour, views
FROM `bigquery-public-data.wikipedia.pageviews_2025`
WHERE DATE(datehour) = DATE '2025-03-30'
  AND wiki = 'en'
"""

review_query("A — full Wikipedia table (estimate only)", WIKI_ALL)
review_query("B — one day (estimate only)", WIKI_DAY)
review_query("C — one day + English wiki (estimate only)", WIKI_DAY_EN)


## 5. `GROUP BY` and JOIN cardinality

BigQuery supports aggregation and JOINs, much like MySQL. Be mindful of the **grain** of each table: duplicate join keys on both sides can multiply the number of output rows.

The first query groups Chicago taxi trips by payment type. **Dry-run only** unless you have reviewed its processing estimate and intentionally chosen a suitable cap. The second query joins tiny in-memory SQL tables (no public-table scan), so it is suitable for an optional protected execution.

**Exercise:** predict the JOIN output. What happens if you add a second row with `zone = 'A'` to `zones`?


In [ ]:
TAXI_GROUP = """
SELECT payment_type, COUNT(*) AS trip_count
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
WHERE trip_start_timestamp >= TIMESTAMP '2025-01-01 00:00:00+00'
  AND trip_start_timestamp < TIMESTAMP '2025-02-01 00:00:00+00'
GROUP BY payment_type
ORDER BY trip_count DESC
"""

review_query("GROUP BY taxi payment type (estimate only)", TAXI_GROUP)


In [ ]:
JOIN_SQL = """
WITH trips AS (
  SELECT 1 AS trip_id, 'A' AS zone UNION ALL
  SELECT 2, 'A' UNION ALL
  SELECT 3, 'B'
), zones AS (
  SELECT 'A' AS zone, 'Central' AS label UNION ALL
  SELECT 'B', 'North'
)
SELECT t.trip_id, z.label
FROM trips AS t
JOIN zones AS z USING (zone)
"""

join_rows = review_query("Tiny JOIN (optional protected execution)", JOIN_SQL, execute=True)
if join_rows is not None:
    print([dict(row) for row in join_rows])


## 6. Execute BigQuery queries from Python

The next query returns a constant string without scanning a public table. Its SQL is defined in Python, then passed to the same helper for review, dry run, and (if both switches are enabled) protected execution. **The default settings execute nothing remotely.**


In [ ]:
HELLO_SQL = "SELECT 'hello' AS message"

hello_rows = review_query("Hello from BigQuery", HELLO_SQL, execute=True)
if hello_rows is not None:
    print([dict(row) for row in hello_rows])


### 6.1. Convert query results to pandas

Use the official client to convert the result of a **protected** query into a DataFrame. This example reuses the tiny `HELLO_SQL` query and the helper above, rather than calling `client.query()` without a dry run.

`pandas-gbq` also provides `pd.read_gbq()` and `df.to_gbq()`, but these do not automatically use our safety helper. We do **not** upload DataFrames or create tables in this optional activity: that could introduce storage costs.


In [ ]:
# Optional: run only after configuring ADC and enabling both switches.
# The helper estimates first and enforces the per-query maximum on execution.
if RUN_DRY_RUNS and RUN_QUERIES:
    import pandas as pd

    protected_rows = run_query_safely(client, HELLO_SQL)
    df = pd.DataFrame([dict(row) for row in protected_rows])
    display(df)
else:
    print("Pandas example skipped. Read the code without connecting to Google Cloud.")


## 7. Quotas, billing, and further learning

Google Cloud has service quotas, project settings, and billing controls. These are **not interchangeable**. A dry run is an estimate; `maximum_bytes_billed` is a per-query on-demand safeguard; budgets/alerts notify but generally do **not** stop charges. Cached results, clustered scans and pricing models may affect actual billed bytes and charges.

- [BigQuery pricing and free allowances](https://cloud.google.com/bigquery/pricing)
- [BigQuery quotas and limits](https://cloud.google.com/bigquery/quotas)
- [Query cost control](https://cloud.google.com/bigquery/docs/best-practices-costs)
- [Billing budgets and alerts](https://cloud.google.com/billing/docs/how-to/budgets)
- [Application Default Credentials](https://cloud.google.com/docs/authentication/provide-credentials-adc)

**Re-check these pages before every new bootcamp cohort:** quotas, prices, interfaces, public datasets, and APIs can change.

## 8. Take-home message

- **OLTP vs OLAP:** MySQL and BigQuery are designed for different primary workloads.
- **Column pruning:** select only the columns needed; `LIMIT` alone is generally not a cost optimization.
- **Partitioning and clustering:** filters can reduce data scanned, depending on table layout.
- **JOINs:** check key uniqueness and row granularity to avoid unintended multiplication.
- **Cost safety:** estimate first, review the result, then execute only when appropriate with a per-query limit.
- **Security:** keep ADC credentials out of notebooks, outputs and Git history.

**Technical nuance:** BigQuery supports `NOT NULL` columns and declarative primary/foreign key constraints (the latter are **not enforced**). Do not assume every field must be nullable or that relational joins are unavailable.

**Optional reflection:** Which of the three techniques—column selection, partition filtering, clustering filtering—produced the greatest reduction in estimated bytes for the public tables you inspected? Why might this change over time?

## Credits and provenance

Original tutorial: **Thomas Belhalfaoui**, created for **Ironhack**. This version has been revised for optional independent self-learning, with additional account-security and query-cost safeguards.

The original repository's licensing and reuse conditions should be confirmed with the course owner before external redistribution. This credit does not itself establish permission to reuse or adapt the material.
